# Machine-Learning Emulators (aemu) — documentation examples

Companion notebook to the **Machine-Learning Emulators (aemu)** documentation
page (`docs/source/aemu_integration.rst`). One section per code snippet /
figure; figure sections regenerate the `tpayne_*` / `mn_*` images in
`docs/img/` (light and dark variants).

Requires the `aemu` extra (`pip install "stellar-spice[aemu]"`); downloading
a pretrained bundle needs network access, so this notebook is shipped
unexecuted.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

from pathlib import Path
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm

DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Render make_fig() in light and dark styles into docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")


## Load a pretrained intensity bundle

The name may be a Hugging Face repo id or a local bundle directory.

In [ ]:
from spice.spectrum import IntensityPretrainedAemuSpectrumEmulator

emu = IntensityPretrainedAemuSpectrumEmulator("RozanskiT/TPayne-spice-small-random")
print(emu.stellar_parameter_names)

## Spectrum of a rotating star

Figure: `tpayne_spectrum_rotation.png` / `tpayne_spectrum_rotation_dark.png`

In [ ]:
from spice.models import IcosphereModel
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.spectrum import simulate_observed_flux

m = IcosphereModel.construct(1000, 1., 1.,
                             emu.to_parameters(dict(logteff=jnp.log10(7000), logg=4.3)),
                             emu.stellar_parameter_names)

mt = evaluate_rotation(add_rotation(m, 100, jnp.array([0., 1., 0.])), 0.)

vws = np.linspace(4670, 4960, 2000)
spec_no_rot = simulate_observed_flux(emu.intensity, m, jnp.log10(vws))
spec_rot = simulate_observed_flux(emu.intensity, mt, jnp.log10(vws))

def rotation_spectrum_fig():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(vws, spec_no_rot[:, 0], color='black', linewidth=1, label='No rotation')
    ax.plot(vws, spec_rot[:, 0], color='royalblue', linewidth=3, label='100 km/s')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Flux [erg/s/cm$^2$/$\AA$]')
    ax.legend()
    return fig

save_doc_fig("tpayne_spectrum_rotation", rotation_spectrum_fig)

## A rotating star with a manganese spot

Figures: `mn_spot_0.png`, `mn_spot_50.png` (+`_dark`)

In [ ]:
from spice.models.spots import add_spot
from spice.plots import plot_3D

timestamps = np.linspace(0, 48*3600, 100)

mn_index = emu.stellar_parameter_names.index('Mn')
m_spotted = add_spot(m, spot_center_theta=1., spot_center_phi=1., spot_radius=30.,
                     parameter_delta=5.0, parameter_index=mn_index)
m_spotted = [evaluate_rotation(add_rotation(m_spotted, 25.), t) for t in timestamps]

save_doc_fig("mn_spot_0",
             lambda: plot_3D(m_spotted[0], property_label='Mn abundance', property=mn_index)[0])
save_doc_fig("mn_spot_50",
             lambda: plot_3D(m_spotted[50], property_label='Mn abundance', property=mn_index)[0])

## Line profiles across the rotation

Figure: `mn_line_profile.png` / `mn_line_profile_dark.png`

In [ ]:
vws = np.linspace(4762, 4769, 2000)
spec_rot_spotted = [simulate_observed_flux(emu.intensity, _m, jnp.log10(vws))
                    for _m in tqdm(m_spotted)]

def line_profile_fig():
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.cool(np.linspace(0, 1, len(spec_rot_spotted)))
    for i, spectrum in enumerate(spec_rot_spotted):
        ax.plot(vws, spectrum[:, 0], color=colors[i], linewidth=1, alpha=0.5)
    sm = plt.cm.ScalarMappable(cmap=plt.cm.cool,
                               norm=plt.Normalize(vmin=0, vmax=timestamps[-1]/3600))
    fig.colorbar(sm, ax=ax, label='Time [h]')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Flux [erg/s/cm$^2$/$\AA$]')
    return fig

save_doc_fig("mn_line_profile", line_profile_fig)

## Line profiles of a pulsating star

Figure: `tpayne_pulsation.png` / `tpayne_pulsation_dark.png`

In [ ]:
from spice.models.mesh_transform import add_pulsation, evaluate_pulsations
import cmasher as cmr

mp = add_pulsation(m, 0, 0, 5., jnp.array([[1e-4, 0.]]))  # period in days

TIMESTAMPS = jnp.linspace(0., 5., 20)
mps = [evaluate_pulsations(mp, t) for t in tqdm(TIMESTAMPS)]
specs = [simulate_observed_flux(emu.intensity, _m, jnp.log10(vws)) for _m in tqdm(mps)]

def pulsation_profiles_fig():
    cmap = cmr.bubblegum
    norm = plt.Normalize(TIMESTAMPS.min(), TIMESTAMPS.max())
    fig, ax = plt.subplots()
    for spec, timestamp in zip(specs, TIMESTAMPS):
        ax.plot(vws, spec[:, 0], color=cmap(norm(timestamp)))
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, ticks=TIMESTAMPS)
    cbar.set_label('Time [d]')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Intensity [erg/s/cm$^2$/$\AA$]')
    ax.tick_params(axis='x', rotation=45)
    return fig

save_doc_fig("tpayne_pulsation", pulsation_profiles_fig)